In [10]:
!pip install ptflops -q

In [11]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define Standard ImageNet Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 2. Set Paths (Update 'extract_path' if you extracted it to a different location in Colab)
extract_path = '/content/drive/MyDrive/datasets'
data_dir = os.path.join(extract_path, 'Skin cancer ISIC The International Skin Imaging Collaboration')
train_dir = os.path.join(data_dir, 'Train')
test_dir = os.path.join(data_dir, 'Test')

# 3. Initialize ImageFolder Datasets
train_data = datasets.ImageFolder(root=train_dir, transform=transform)
test_data = datasets.ImageFolder(root=test_dir, transform=transform)

# 4. Create DataLoaders
# batch_size=32 is standard, but reduce it to 16 if you run into CUDA Out of Memory errors
train_loader = DataLoader(train_data, batch_size=32, shuffle=True,pin_memory=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False,pin_memory=True, num_workers=0)

print(f"Loaded {len(train_data.classes)} classes: {train_data.classes}")
print(f"Training images: {len(train_data)} | Testing images: {len(test_data)}")

Loaded 9 classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Training images: 2239 | Testing images: 118


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import time
import gc
from ptflops import get_model_complexity_info
from tqdm import tqdm # Added for progress tracking

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 9
num_epochs = 5 # Increased for meaningful metric comparisons

model_configs = {
    'AlexNet': (models.alexnet, models.AlexNet_Weights.IMAGENET1K_V1),
    'VGG16': (models.vgg16, models.VGG16_Weights.IMAGENET1K_V1),
    'VGG19': (models.vgg19, models.VGG19_Weights.IMAGENET1K_V1),
    'ResNet18': (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
    'ResNet50': (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1),
    'ResNet101': (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V1),
    'DenseNet121': (models.densenet121, models.DenseNet121_Weights.IMAGENET1K_V1),
    'EfficientNet-B0': (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1)
}

for model_name, (model_func, weights) in model_configs.items():
    print(f"\n{'='*50}\nEvaluating {model_name}\n{'='*50}")

    model = model_func(weights=weights)

    # Modify Final Classification Layer
    if hasattr(model, 'fc'):
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif hasattr(model, 'classifier'):
        if isinstance(model.classifier, nn.Sequential):
            num_ftrs = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(num_ftrs, num_classes)
        else:
            num_ftrs = model.classifier.in_features
            model.classifier = nn.Linear(num_ftrs, num_classes)

    model = model.to(device)

    # Table 3: Computational Efficiency
    macs, params = get_model_complexity_info(model, (3, 224, 224), as_strings=False, print_per_layer_stat=False, verbose=False)
    flops_g = (macs * 2) / 1e9
    params_m = params / 1e6
    model_size_mb = params * 4 / (1024 ** 2)

    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    model.eval()
    with torch.no_grad():
        for _ in range(10): model(dummy_input)
        start_time = time.time()
        for _ in range(50): model(dummy_input)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        end_time = time.time()
    infer_time_ms = ((end_time - start_time) / 50) * 1000

    print(f"[Table 3] Params: {params_m:.2f}M | Size: {model_size_mb:.2f}MB | FLOPs: {flops_g:.2f}G | Inference: {infer_time_ms:.2f}ms")

    # Training Loop
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    # Added StepLR to decay learning rate by 0.1 every 3 epochs for better fine-tuning
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        # Wrapped train_loader in tqdm for a visual progress bar
        train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

        for inputs, labels in train_iterator:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

            # Update progress bar with current batch loss
            train_iterator.set_postfix({'Loss': f"{loss.item():.4f}"})

        scheduler.step()
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs} Completed - Average Loss: {epoch_loss:.4f}")

    # Table 1: Evaluation Metrics
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        # Added progress bar for testing
        for inputs, labels in tqdm(test_loader, desc="Evaluating", leave=False):
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr') * 100
    except ValueError:
        auc = 0.0

    print(f"[Table 1] Acc: {acc:.2f}% | Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc:.2f}%")

    # Strict Memory Management
    del model, optimizer, criterion, scheduler
    gc.collect()
    torch.cuda.empty_cache()


Evaluating AlexNet
[Table 3] Params: 57.04M | Size: 217.59MB | FLOPs: 1.42G | Inference: 2.45ms


Epoch 1/5 Completed - Average Loss: 1.3997


Epoch 2/5 Completed - Average Loss: 0.9043


Epoch 3/5 Completed - Average Loss: 0.7189


Epoch 4/5 Completed - Average Loss: 0.4697


...

Epoch 2/5 Completed - Average Loss: 1.1087


Epoch 3/5 Completed - Average Loss: 0.7833


Epoch 4/5 Completed - Average Loss: 0.5992


Epoch 5/5 Completed - Average Loss: 0.5912


[Table 1] Acc: 62.71% | Prec: 59.21% | Rec: 60.42% | F1: 55.68% | AUC: 91.75%


In [14]:
!pip install xgboost -q

In [15]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
from tqdm import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize Feature Extractor (Using ResNet50 as the baseline)
print("Setting up Deep Feature Extractor...")
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
# Replace the final fully connected layer with an Identity layer to output raw features
feature_extractor.fc = nn.Identity()
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Helper function to process dataset into 1D feature vectors
def extract_features(dataloader):
    features_list = []
    labels_list = []
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Extracting", leave=False):
            inputs = inputs.to(device)
            features = feature_extractor(inputs)
            features_list.extend(features.cpu().numpy())
            labels_list.extend(labels.numpy())
    return np.array(features_list), np.array(labels_list)

# 2. Extract Features for Train and Test Sets
print("Extracting training features...")
X_train, y_train = extract_features(train_loader)
print("Extracting testing features...")
X_test, y_test = extract_features(test_loader)

# Clear GPU memory since we only need the CPU for traditional ML classifiers now
del feature_extractor
gc.collect()
torch.cuda.empty_cache()

# 3. Define Classifiers for Table 2
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    # probability=True is required for SVMs to calculate AUC
    'Linear SVM': SVC(kernel='linear', probability=True),
    'RBF-SVM': SVC(kernel='rbf', probability=True),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

print(f"\n{'='*50}\nTraining Classifiers for Table 2\n{'='*50}")

# 4. Train and Evaluate Each Classifier
for name, clf in classifiers.items():
    print(f"\nTraining {name}...")

    # Fit the classifier
    clf.fit(X_train, y_train)

    # Predict classes and probabilities
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)

    # Calculate Metrics
    acc = accuracy_score(y_test, preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

    try:
        auc = roc_auc_score(y_test, probs, multi_class='ovr') * 100
    except ValueError:
        auc = 0.0

    print(f"[Table 2: {name}]")
    print(f"Acc: {acc:.2f}% | Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc:.2f}%")

Setting up Deep Feature Extractor...
Extracting training features...


Extracting testing features...



Training Classifiers for Table 2

Training Logistic Regression...
[Table 2: Logistic Regression]
Acc: 46.61% | Prec: 44.02% | Rec: 47.22% | F1: 42.72% | AUC: 87.71%

Training Decision Tree...
[Table 2: Decision Tree]
Acc: 36.44% | Prec: 30.76% | Rec: 35.88% | F1: 30.63% | AUC: 63.86%

Training Random Forest...
[Table 2: Random Forest]
Acc: 38.14% | Prec: 36.04% | Rec: 40.28% | F1: 30.98% | AUC: 80.45%

Training K-Nearest Neighbors (KNN)...
[Table 2: K-Nearest Neighbors (KNN)]
Acc: 40.68% | Prec: 42.79% | Rec: 42.36% | F1: 37.20% | AUC: 76.36%

Training Linear SVM...
[Table 2: Linear SVM]
Acc: 43.22% | Prec: 41.88% | Rec: 44.44% | F1: 39.24% | AUC: 89.16%

Training RBF-SVM...
[Table 2: RBF-SVM]
Acc: 47.46% | Prec: 51.85% | Rec: 47.92% | F1: 42.92% | AUC: 90.05%

Training XGBoost...


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [06:24:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[Table 2: XGBoost]
Acc: 44.92% | Prec: 45.24% | Rec: 45.83% | F1: 39.81% | AUC: 79.57%
